# Phase 1 — Load, clean, and **audit** the data

**Why audit before modelling?** (paper §3.3) A model is only as trustworthy as its
data. Before training we check three things:

1. **Duplicates / redundancy** — street photos taken seconds apart look identical; if
   near-duplicates land on both sides of a split, the model can "cheat" by recognising
   a scene it already saw. We measure how much redundancy exists.
2. **Images per station** — decides whether our leakage-safe split (grouping by
   station) is feasible.
3. **A physics sanity check** — hazier photos should have lower transmission and higher
   AQI; if not, images and labels were mis-paired.

We also **clean** the data: drop dead columns, remove duplicate rows, and shuffle.

## Bootstrap — run this first

This one cell makes the notebook self-contained: it grabs the code from GitHub (if it
isn't already here), installs the libraries, connects Google Drive, and makes our `src`
modules importable. **Set `REPO_URL` to your repository's URL.** It's safe to re-run and
also works on a laptop.

In [ ]:
# === Bootstrap — RUN ME FIRST (set REPO_URL to your repo) ===
REPO_URL = "https://github.com/YOUR_USERNAME/pm25-visual-aq.git"   # <-- EDIT THIS

import os, sys, subprocess

def _find_repo_root():
    # Are we already inside the repo (or just above the notebooks/ folder)?
    for cand in (".", "..", "pm25-visual-aq"):
        if os.path.isdir(os.path.join(cand, "src")):
            return os.path.abspath(cand)
    return None

_root = _find_repo_root()
if _root is None:                       # fresh Colab session: clone the code
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "pm25-visual-aq"], check=True)
    _root = os.path.abspath("pm25-visual-aq")
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

print("repo root:", _root, "| Colab:", IN_COLAB)

Now import our modules and pick the data source.

In [ ]:
from src.config import load_config
from src import data, audit
cfg = load_config()

# Full dataset (Colab). For a quick laptop test, set SOURCE = "tests/fixture_ds".
SOURCE = cfg["data"]["drive_path"]
print("Loading from:", SOURCE)

## Load + clean

`load_clean` pools the train/test splits into one pool (we make our own splits in
Phase 2), drops columns with no signal, removes duplicate `image_id` rows, and
shuffles with a fixed seed so file ordering can't bias a split.

In [ ]:
ds, df = data.load_clean(SOURCE, from_disk=True, seed=cfg["seed"])
print("rows after cleaning:", len(df))
print("duplicate rows removed:", df.attrs.get("n_duplicates_removed"))
df.head()

## The label: an **AQI index**, not µg/m³

PM25Vision's label is a US-EPA Air Quality Index value (~1–530), a unitless index —
*not* a raw concentration. Every error we report is in **AQI points**. The histogram is
right-skewed, which is why we later train on `log(AQI)`.

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7,3))
plt.hist(df["pm25"], bins=50)
plt.xlabel("pm25 (AQI index)"); plt.ylabel("number of photos")
plt.title("Label distribution — right-skewed"); plt.show()
print(df["pm25"].describe())

## Images per station

Decides whether we can split *by station* (our leakage-safe protocol). With thousands
of stations, most contributing only a couple of images, grouping is easy. The few busy
stations are where near-duplicate risk concentrates.

In [ ]:
sps = audit.images_per_station(df, station_col=cfg["data"]["station_col"])
print("stations: %d | images/station  median=%.1f  mean=%.2f  max=%d  (%.0f%% have just 1)"
      % (sps["n_stations"], sps["median"], sps["mean"], sps["max"], sps["pct_single_image"]))
plt.figure(figsize=(7,3))
plt.hist(sps["counts"].values, bins=40)
plt.xlabel("images at one station"); plt.ylabel("number of stations")
plt.title("Most stations contribute only a few images"); plt.show()

## Near-duplicate check (perceptual hashing)

A *perceptual hash* is a fingerprint where **similar images get similar fingerprints**.
We fingerprint every image and group ones whose fingerprints differ by only a few bits.
The **distinct-ratio** = groups ÷ images: 1.0 means no near-duplicates; lower means
redundancy we must keep out of the split.

> ⏳ On the full dataset this decodes ~11k images and takes a few minutes.

In [ ]:
rep = audit.redundancy_report(ds, df, hash_size=8, max_distance=5)
print("images: %d | distinct groups: %d | distinct-ratio: %.3f | largest group: %d"
      % (rep["n_images"], rep["n_distinct_groups"], rep["distinct_ratio"], rep["largest_group"]))

## Physics falsification test

Physics says hazier photos have **lower transmission** and **higher AQI**, so average
transmission should be **negatively** correlated with the label. A clearly negative
correlation means images and labels line up.

In [ ]:
cor = audit.transmission_label_correlation(
    ds, df, target_col=cfg["data"]["target_col"], sample=1000, seed=cfg["seed"])
print("Pearson r = %.3f (p=%.1e)  |  Spearman r = %.3f  |  negative as expected? %s"
      % (cor["pearson_r"], cor["pearson_p"], cor["spearman_r"], cor["passes"]))
plt.figure(figsize=(5,4))
plt.scatter(cor["mean_transmission"], cor["labels"], s=6, alpha=0.4)
plt.xlabel("average transmission (clearer →)"); plt.ylabel("AQI label")
plt.title("Should slope downward"); plt.show()

## Where in the world are these photos?

Coverage skews to East Asia, Europe, and India, with little in the Americas or Africa.
That's fine, but any "generalises everywhere" claim must be scoped to the regions
actually represented (paper §3.2).

In [ ]:
plt.figure(figsize=(8,4))
plt.scatter(df[cfg["data"]["lon_col"]], df[cfg["data"]["lat_col"]], s=4, alpha=0.3)
plt.xlabel("longitude"); plt.ylabel("latitude"); plt.title("Station geography"); plt.show()

## What we found (summary)

- Label is **AQI (1–530)**, right-skewed → we'll train on `log(AQI)`.
- **3,261 stations**, most with only a few images → station-grouped split is easy.
- Near-duplicate redundancy is concentrated in a handful of busy stations.
- The physics check should be **negative** — confirming images and labels match.
- Geography is skewed → we scope our claims honestly.

**Next:** `02_splits.ipynb` — leakage-safe train/calibration/test splits, and measuring
how much a naive random split inflates results.